# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

### Selected method

This notebook treats the task as a supervised binary classification and ranking problem. The target indicates whether a content item experienced a decline in recent search performance.

I start with Logistic Regression because it provides a simple and interpretable learned benchmark. Its coefficients help show the direction in which each feature contributes to the estimated probability of decline.

I also train a Random Forest as a second model. It can capture nonlinear relationships and interactions that Logistic Regression may miss. However, the more complex model will only be preferred if it produces a meaningful improvement on the same validation split and metrics.

The model probabilities are used as ranking scores. This matches the operational goal of prioritizing a limited number of content items for human review rather than automatically classifying every page.

Both models will be compared with the Week-4 hand-written baseline using the same test rows, the same target, and the same ranking metrics. The main comparison metric is Precision@50, with Precision@20, average precision, ROC-AUC, and the target base rate reported as supporting metrics.

The results are interpreted as decision support. They do not show that any feature causes a decline or that reviewing a recommended page will necessarily improve performance.


In [3]:
# Core imports and reproducibility settings

from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd

from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Python version:", sys.version.split()[0])
print("Pandas version:", pd.__version__)
print("Random seed:", RANDOM_STATE)


Python version: 3.12.13
Pandas version: 2.2.2
Random seed: 42


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

### Split design

The train/test split is performed at the client level using GroupShuffleSplit.

Grouping by client prevents pages from the same client appearing in both the training and test sets. This reduces information leakage and produces a more realistic estimate of how the model would generalize to unseen clients.

The split uses approximately 80% of the clients for training and 20% for testing. Both the learned models and the Week-4 baseline are evaluated on exactly the same test set to ensure a fair comparison.


In [5]:
from pathlib import Path
import os

REPO_URL = "https://github.com/muhammetalicvs-prog/flyrank-ml.git"
REPO_DIR = Path("/content/flyrank-ml")

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("Repository already exists:", REPO_DIR)

os.chdir(REPO_DIR)

print("Current directory:", Path.cwd())
print("Data file exists:", Path("data/raw/content_refresh_anonymized.csv").exists());

Cloning into '/content/flyrank-ml'...
remote: Enumerating objects: 128, done.
remote: Counting objects: 100% (128/128), done.
remote: Compressing objects: 100% (99/99), done.
remote: Total 128 (delta 41), reused 78 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (128/128), 1.88 MiB | 10.59 MiB/s, done.
Resolving deltas: 100% (41/41), done.
Current directory: /content/flyrank-ml
Data file exists: True


In [6]:
# Load the prepared dataset and create a leakage-safe grouped split

candidate_paths = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("/content/flyrank-ml/data/raw/content_refresh_anonymized.csv"),
    Path("/content/flyrank-ml/flyrank-ml/data/raw/content_refresh_anonymized.csv"),
]

DATA_PATH = next(
    (path for path in candidate_paths if path.exists()),
    None,
)

if DATA_PATH is None:
    raise FileNotFoundError(
        "Prepared dataset was not found. "
        "Make sure the repository is cloned and the current directory is the repo root."
    )

df = pd.read_csv(DATA_PATH)

print("Dataset path:", DATA_PATH.resolve())
print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")

required_columns = {
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
}

missing_columns = required_columns.difference(df.columns)

if missing_columns:
    raise ValueError(
        f"Required columns are missing: {sorted(missing_columns)}"
    )

# The target is derived only for evaluation.
# It is never included in the model features.
df["is_declining_label"] = (
    df["trend_direction"]
    .astype(str)
    .str.lower()
    .eq("down")
    .astype(int)
)

# Safe features available before the outcome label
numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
]

categorical_features = [
    "competition_level",
    "content_type",
    "main_intent",
    "provider_used",
    "model_used",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "char_count_tier",
    "impression_tier",
    "position_tier",
]

feature_columns = numeric_features + categorical_features
target_column = "is_declining_label"
group_column = "client_id"

# Explicit leakage checks
forbidden_features = {
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "content_id",
    "client_id",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
}

assert not forbidden_features.intersection(feature_columns), (
    "Leakage or identifier columns were included in the features."
)

model_df = df[
    ["content_id", group_column, target_column] + feature_columns
].copy()

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE,
)

train_idx, test_idx = next(
    splitter.split(
        model_df[feature_columns],
        model_df[target_column],
        groups=model_df[group_column],
    )
)

train_df = model_df.iloc[train_idx].copy()
test_df = model_df.iloc[test_idx].copy()

X_train = train_df[feature_columns]
y_train = train_df[target_column]

X_test = test_df[feature_columns]
y_test = test_df[target_column]

train_clients = set(train_df[group_column])
test_clients = set(test_df[group_column])

print("\nSplit summary")
print("-" * 40)
print(f"Training rows: {len(train_df):,}")
print(f"Test rows: {len(test_df):,}")
print(f"Training clients: {len(train_clients):,}")
print(f"Test clients: {len(test_clients):,}")
print(f"Client overlap: {len(train_clients.intersection(test_clients))}")
print(f"Training decline rate: {y_train.mean():.2%}")
print(f"Test decline rate: {y_test.mean():.2%}")
print(f"Numeric features: {len(numeric_features)}")
print(f"Categorical features: {len(categorical_features)}")

assert train_clients.isdisjoint(test_clients)
assert X_train.columns.tolist() == X_test.columns.tolist()
assert target_column not in feature_columns
assert "trend_direction" not in feature_columns
assert "trend_pct" not in feature_columns

Dataset path: /content/flyrank-ml/data/raw/content_refresh_anonymized.csv
Rows: 30,000
Columns: 44

Split summary
----------------------------------------
Training rows: 23,837
Test rows: 6,163
Training clients: 25
Test clients: 7
Client overlap: 0
Training decline rate: 55.01%
Test decline rate: 51.10%
Numeric features: 22
Categorical features: 11


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

### Training and fair baseline comparison

The preprocessing steps are learned only from the training set. Missing numeric values are filled with the training-set median and then standardized for Logistic Regression. Missing categorical values are filled with the most frequent training value and encoded using one-hot encoding.

Logistic Regression is used as the main interpretable model. Random Forest is included as a nonlinear comparison, but it will not be preferred merely because it is more complex.

The Week-4 baseline is recreated using its original visibility, position, and low-CTR components. The baseline, Logistic Regression, and Random Forest are all ranked and evaluated on the same grouped test set.

For the baseline, rows that do not satisfy the original Week-4 eligibility rule receive a score below all eligible rows. This preserves the original rule while allowing ranking metrics to be calculated over the complete test set.


In [7]:
# Build preprocessing pipelines, train models, and recreate the Week-4 baseline

numeric_preprocessor = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_preprocessor = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True,
            ),
        ),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_preprocessor, numeric_features),
        ("categorical", categorical_preprocessor, categorical_features),
    ],
    remainder="drop",
)

logistic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                max_iter=2000,
                class_weight=None,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

random_forest_model = Pipeline(
    steps=[
        (
            "preprocessor",
            ColumnTransformer(
                transformers=[
                    (
                        "numeric",
                        Pipeline(
                            steps=[
                                ("imputer", SimpleImputer(strategy="median")),
                            ]
                        ),
                        numeric_features,
                    ),
                    (
                        "categorical",
                        categorical_preprocessor,
                        categorical_features,
                    ),
                ],
                remainder="drop",
            ),
        ),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=300,
                max_depth=10,
                min_samples_leaf=10,
                max_features="sqrt",
                class_weight=None,
                n_jobs=-1,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

print("Training Logistic Regression...")
logistic_model.fit(X_train, y_train)

print("Training Random Forest...")
random_forest_model.fit(X_train, y_train)

logistic_scores = logistic_model.predict_proba(X_test)[:, 1]
random_forest_scores = random_forest_model.predict_proba(X_test)[:, 1]

# Recreate the Week-4 baseline on exactly the same test rows
baseline_test = test_df[
    [
        "content_id",
        "client_id",
        target_column,
        "impressions_90d",
        "ctr",
        "avg_position",
    ]
].copy()

baseline_test["visibility_score"] = np.clip(
    np.log1p(baseline_test["impressions_90d"].clip(lower=0))
    / np.log1p(10000),
    0,
    1,
)

baseline_test["position_score"] = np.clip(
    (20 - baseline_test["avg_position"]) / 20,
    0,
    1,
)

baseline_test["low_ctr_score"] = np.clip(
    (0.20 - baseline_test["ctr"]) / 0.20,
    0,
    1,
)

baseline_test["raw_action_score"] = (
    100
    * (
        0.40 * baseline_test["visibility_score"]
        + 0.30 * baseline_test["position_score"]
        + 0.30 * baseline_test["low_ctr_score"]
    )
)

baseline_test["baseline_eligible"] = (
    (baseline_test["impressions_90d"] >= 100)
    & (baseline_test["avg_position"] > 0)
    & (baseline_test["avg_position"] <= 20)
    & baseline_test["ctr"].notna()
    & (baseline_test["ctr"] < 0.20)
)

# Eligible rows rank above ineligible rows.
# The raw score still orders rows within each group.
baseline_scores = np.where(
    baseline_test["baseline_eligible"],
    1000 + baseline_test["raw_action_score"],
    baseline_test["raw_action_score"],
)

assert len(logistic_scores) == len(y_test)
assert len(random_forest_scores) == len(y_test)
assert len(baseline_scores) == len(y_test)

print("\nTraining complete")
print("-" * 40)
print(f"Test rows scored: {len(y_test):,}")
print(
    "Baseline-eligible test rows:",
    f"{baseline_test['baseline_eligible'].sum():,}",
)
print(
    "Baseline-eligible share:",
    f"{baseline_test['baseline_eligible'].mean():.2%}",
)
print(
    "Logistic score range:",
    f"{logistic_scores.min():.4f} to {logistic_scores.max():.4f}",
)
print(
    "Random Forest score range:",
    f"{random_forest_scores.min():.4f} to {random_forest_scores.max():.4f}",
)

Training Logistic Regression...
Training Random Forest...

Training complete
----------------------------------------
Test rows scored: 6,163
Baseline-eligible test rows: 1,388
Baseline-eligible share: 22.52%
Logistic score range: 0.0324 to 0.9319
Random Forest score range: 0.0545 to 0.8921


In [8]:
# Compare every method on the same test rows and the same metrics

def precision_at_k(y_true, scores, k):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    k = min(k, len(y_true))
    top_indices = np.argsort(-scores, kind="stable")[:k]

    return float(y_true[top_indices].mean())


def recall_at_k(y_true, scores, k):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    total_positives = y_true.sum()

    if total_positives == 0:
        return np.nan

    k = min(k, len(y_true))
    top_indices = np.argsort(-scores, kind="stable")[:k]

    return float(y_true[top_indices].sum() / total_positives)


def evaluate_ranking_method(name, y_true, scores):
    return {
        "method": name,
        "precision_at_20": precision_at_k(y_true, scores, 20),
        "precision_at_50": precision_at_k(y_true, scores, 50),
        "recall_at_50": recall_at_k(y_true, scores, 50),
        "average_precision": average_precision_score(y_true, scores),
        "roc_auc": roc_auc_score(y_true, scores),
    }


comparison_table = pd.DataFrame(
    [
        evaluate_ranking_method(
            "Week-4 baseline",
            y_test,
            baseline_scores,
        ),
        evaluate_ranking_method(
            "Logistic Regression",
            y_test,
            logistic_scores,
        ),
        evaluate_ranking_method(
            "Random Forest",
            y_test,
            random_forest_scores,
        ),
    ]
)

comparison_table["test_base_rate"] = y_test.mean()

metric_columns = [
    "precision_at_20",
    "precision_at_50",
    "recall_at_50",
    "average_precision",
    "roc_auc",
    "test_base_rate",
]

comparison_table[metric_columns] = comparison_table[
    metric_columns
].round(4)

comparison_table = comparison_table.sort_values(
    by="precision_at_50",
    ascending=False,
).reset_index(drop=True)

display(comparison_table)

best_method = comparison_table.loc[0, "method"]
best_precision_50 = comparison_table.loc[0, "precision_at_50"]

baseline_precision_50 = comparison_table.loc[
    comparison_table["method"].eq("Week-4 baseline"),
    "precision_at_50",
].iloc[0]

print(f"Best method by Precision@50: {best_method}")
print(f"Best Precision@50: {best_precision_50:.2%}")
print(f"Baseline Precision@50: {baseline_precision_50:.2%}")
print(
    "Absolute improvement over baseline:",
    f"{best_precision_50 - baseline_precision_50:+.2%}",
)

,method,precision_at_20,precision_at_50,recall_at_50,average_precision,roc_auc,test_base_rate
0,Week-4 baseline,0.8,0.88,0.0140,0.5989,0.5618,0.511
1,Logistic Regression,0.7,0.64,0.0102,0.5691,0.5777,0.511
2,Random Forest,0.6,0.58,0.0092,0.5898,0.6111,0.511


Best method by Precision@50: Week-4 baseline
Best Precision@50: 88.00%
Baseline Precision@50: 88.00%
Absolute improvement over baseline: +0.00%


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [9]:
# Inspect top-ranked errors and correct recommendations

evaluation_df = test_df[
    [
        "content_id",
        "client_id",
        target_column,
        "impressions_90d",
        "clicks_90d",
        "ctr",
        "avg_position",
        "engagement_rate",
        "days_since_last_update",
    ]
].copy()

evaluation_df["baseline_score"] = baseline_scores
evaluation_df["logistic_score"] = logistic_scores
evaluation_df["random_forest_score"] = random_forest_scores


def top_k_error_summary(data, score_column, method_name, k=50):
    ranked = (
        data.sort_values(score_column, ascending=False)
        .head(k)
        .copy()
    )

    ranked["prediction_result"] = np.where(
        ranked[target_column].eq(1),
        "true_positive",
        "false_positive",
    )

    summary = pd.DataFrame(
        {
            "method": [method_name],
            "top_k": [k],
            "true_positives": [ranked[target_column].sum()],
            "false_positives": [(ranked[target_column] == 0).sum()],
            "precision_at_k": [ranked[target_column].mean()],
            "median_impressions": [ranked["impressions_90d"].median()],
            "median_ctr": [ranked["ctr"].median()],
            "median_position": [ranked["avg_position"].median()],
            "median_days_since_update": [
                ranked["days_since_last_update"].median()
            ],
        }
    )

    return ranked, summary


baseline_top50, baseline_error_summary = top_k_error_summary(
    evaluation_df,
    "baseline_score",
    "Week-4 baseline",
)

logistic_top50, logistic_error_summary = top_k_error_summary(
    evaluation_df,
    "logistic_score",
    "Logistic Regression",
)

forest_top50, forest_error_summary = top_k_error_summary(
    evaluation_df,
    "random_forest_score",
    "Random Forest",
)

error_summary_table = pd.concat(
    [
        baseline_error_summary,
        logistic_error_summary,
        forest_error_summary,
    ],
    ignore_index=True,
)

display(error_summary_table.round(4))

,method,top_k,true_positives,false_positives,precision_at_k,median_impressions,median_ctr,median_position,median_days_since_update
0,Week-4 baseline,50,44,6,0.88,7566.0,0.015,3.10,20.0
1,Logistic Regression,50,32,18,0.64,1151.0,0.065,7.70,102.0
2,Random Forest,50,29,21,0.58,1181.5,0.070,16.85,104.0


In [10]:
# Examine false positives in the operational top-50 lists

display_columns = [
    "content_id",
    "client_id",
    target_column,
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "engagement_rate",
    "days_since_last_update",
]

print("Week-4 baseline false positives")
display(
    baseline_top50.loc[
        baseline_top50[target_column].eq(0),
        display_columns + ["baseline_score"],
    ]
    .sort_values("baseline_score", ascending=False)
    .head(10)
)

print("\nLogistic Regression false positives")
display(
    logistic_top50.loc[
        logistic_top50[target_column].eq(0),
        display_columns + ["logistic_score"],
    ]
    .sort_values("logistic_score", ascending=False)
    .head(10)
)

print("\nRandom Forest false positives")
display(
    forest_top50.loc[
        forest_top50[target_column].eq(0),
        display_columns + ["random_forest_score"],
    ]
    .sort_values("random_forest_score", ascending=False)
    .head(10)
)

Week-4 baseline false positives


,content_id,client_id,is_declining_label,impressions_90d,clicks_90d,ctr,avg_position,engagement_rate,days_since_last_update,baseline_score
22928,content_929aa622b6a0,client_4e07408562,0,11301,3,0.03,2.4,0.00,104,1091.900000
12869,content_5d5653c4eb4f,client_4e07408562,0,15101,0,0.00,5.7,0.00,7,1091.450000
25560,content_1d2233dc3323,client_f369cb89fc,0,1463,0,0.00,1.5,0.00,8,1089.405067
13631,content_d274ac4158ef,client_4e07408562,0,65138,6,0.01,6.8,4.00,26,1088.300000
11163,content_ceaa28bba4ca,client_4e07408562,0,43287,26,0.06,3.9,2.17,104,1085.150000
16736,content_e12868d1f396,client_4e07408562,0,149712,104,0.07,2.9,5.94,7,1085.150000



Logistic Regression false positives


,content_id,client_id,is_declining_label,impressions_90d,clicks_90d,ctr,avg_position,engagement_rate,days_since_last_update,logistic_score
20736,content_41baf0722ad9,client_8527a891e2,0,3115,0,0.00,12.8,0.00,104,0.911621
10175,content_374e795aab68,client_f369cb89fc,0,235,2,0.85,31.0,0.00,20,0.906780
11887,content_ce59581533ca,client_8527a891e2,0,289,2,0.69,18.8,0.00,102,0.898231
18531,content_d10f9ce1e0cd,client_4e07408562,0,166,1,0.60,16.1,0.00,104,0.897735
26614,content_7be5f150dc65,client_f369cb89fc,0,290,0,0.00,5.9,0.00,20,0.897301
4905,content_f0d98be4b42c,client_4e07408562,0,5818,9,0.15,5.1,22.22,104,0.896786
4050,content_500bd3907331,client_4e07408562,0,4037,4,0.10,5.5,0.00,104,0.888608
12332,content_4d9f36001f06,client_8527a891e2,0,3369,1,0.03,13.2,33.33,104,0.888305
6739,content_f45787e64ac2,client_4e07408562,0,291,1,0.34,5.2,0.00,104,0.885422
11061,content_0b47dae0c7f9,client_8527a891e2,0,1191,0,0.00,23.1,0.00,103,0.880565



Random Forest false positives


,content_id,client_id,is_declining_label,impressions_90d,clicks_90d,ctr,avg_position,engagement_rate,days_since_last_update,random_forest_score
22042,content_2ba626fea4d6,client_8527a891e2,0,360,0,0.00,7.2,0.0,104,0.882247
22526,content_1d0963b56227,client_4e07408562,0,3445,3,0.09,39.0,20.0,104,0.881700
10080,content_35d63627bf3e,client_8527a891e2,0,1525,0,0.00,32.6,0.0,103,0.879676
11061,content_0b47dae0c7f9,client_8527a891e2,0,1191,0,0.00,23.1,0.0,103,0.879419
4050,content_500bd3907331,client_4e07408562,0,4037,4,0.10,5.5,0.0,104,0.876053
12069,content_ff4370afd49c,client_4e07408562,0,1677,3,0.18,33.1,0.0,104,0.872719
22524,content_846bb4dd8b44,client_8527a891e2,0,870,1,0.11,17.6,0.0,104,0.872672
5399,content_6677fd6c4ea5,client_4e07408562,0,1152,2,0.17,34.8,0.0,104,0.869715
29456,content_b46c62b14582,client_8527a891e2,0,6240,8,0.13,31.8,0.0,103,0.865132
20736,content_41baf0722ad9,client_8527a891e2,0,3115,0,0.00,12.8,0.0,104,0.865020


In [11]:
# Estimate feature importance on held-out clients using permutation importance

permutation_result = permutation_importance(
    random_forest_model,
    X_test,
    y_test,
    scoring="roc_auc",
    n_repeats=5,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

permutation_table = pd.DataFrame(
    {
        "feature": feature_columns,
        "importance_mean": permutation_result.importances_mean,
        "importance_std": permutation_result.importances_std,
    }
).sort_values(
    "importance_mean",
    ascending=False,
)

print("Top Random Forest permutation importances")
display(permutation_table.head(15).round(4))

Top Random Forest permutation importances


,feature,importance_mean,importance_std
13,days_with_impressions,0.0360,0.0042
5,impressions_90d,0.0131,0.0019
18,avg_position,0.0070,0.0012
6,clicks_90d,0.0062,0.0005
17,ctr,0.0048,0.0002
31,impression_tier,0.0029,0.0005
32,position_tier,0.0028,0.0010
20,scroll_rate,0.0025,0.0008
14,days_with_sessions,0.0025,0.0003
15,content_age_days,0.0023,0.0016


In [12]:
# Inspect the strongest Logistic Regression coefficients

logistic_preprocessor = logistic_model.named_steps["preprocessor"]
logistic_classifier = logistic_model.named_steps["classifier"]

transformed_feature_names = (
    logistic_preprocessor.get_feature_names_out()
)

coefficient_table = pd.DataFrame(
    {
        "feature": transformed_feature_names,
        "coefficient": logistic_classifier.coef_[0],
    }
)

coefficient_table["absolute_coefficient"] = (
    coefficient_table["coefficient"].abs()
)

print("Features most associated with a higher estimated decline probability")
display(
    coefficient_table
    .sort_values("coefficient", ascending=False)
    .head(10)
    .round(4)
)

print("Features most associated with a lower estimated decline probability")
display(
    coefficient_table
    .sort_values("coefficient", ascending=True)
    .head(10)
    .round(4)
)

Features most associated with a higher estimated decline probability


,feature,coefficient,absolute_coefficient
8,numeric__sessions_90d,0.8594,0.8594
13,numeric__days_with_impressions,0.7808,0.7808
37,categorical__model_used_gpt-5-mini,0.5912,0.5912
27,categorical__content_type_keyword article,0.5246,0.5246
43,categorical__freshness_tier_0-30,0.4361,0.4361
3,numeric__word_count,0.3892,0.3892
47,categorical__word_count_tier_1000-2000,0.3837,0.3837
62,categorical__position_tier_striking,0.3181,0.3181
40,categorical__age_tier_31-90,0.3011,0.3011
61,categorical__position_tier_page_3_5,0.2712,0.2712


Features most associated with a lower estimated decline probability


,feature,coefficient,absolute_coefficient
63,categorical__position_tier_top_3,-1.0304,1.0304
9,numeric__users_90d,-0.9694,0.9694
26,categorical__content_type_feedly article,-0.5441,0.5441
38,categorical__model_used_unknown,-0.5085,0.5085
14,numeric__days_with_sessions,-0.5024,0.5024
30,categorical__main_intent_navigational,-0.4511,0.4511
45,categorical__freshness_tier_31-90,-0.4458,0.4458
15,numeric__content_age_days,-0.3902,0.3902
44,categorical__freshness_tier_181+,-0.3662,0.3662
58,categorical__impression_tier_moderate,-0.3609,0.3609


### Model comparison

The Week-4 baseline achieved the strongest operational ranking result. Its Precision@50 was 0.88, meaning that 44 of its first 50 recommendations belonged to the decline class. Logistic Regression achieved a Precision@50 of 0.64, while Random Forest achieved 0.58.

The learned models therefore did not improve the primary business metric. Increasing model complexity was not rewarded because neither learned model produced a better top-50 review queue than the existing hand-written rule.

Random Forest did achieve the highest ROC-AUC at 0.6111, compared with 0.5777 for Logistic Regression and 0.5618 for the baseline. This suggests that Random Forest was somewhat better at ordering positive and negative examples across the complete test set. However, its advantage did not extend to the highest-priority portion of the ranking.

This distinction matters because the operational task is not to classify every content item. It is to identify a small review queue. For that purpose, top-k precision is more directly useful than overall ROC-AUC.

### Error interpretation

The Week-4 baseline produced 6 false positives among its first 50 recommendations. These rows appeared actionable because they had substantial visibility, relatively favorable search positions, and low CTR, but they did not belong to the observed decline class.

These errors show that low CTR with visibility is not equivalent to future decline. A page may have a low aggregated CTR because of search intent, branded versus non-branded query composition, SERP features, advertisements, or query-level differences that are not represented in this dataset.

The learned models produced more false positives in their top-50 queues. This indicates that the available features and target do not provide enough stable predictive signal for the models to outperform the focused business rule at the top of the ranking.

The test decline rate was 0.511. All three methods performed above this base rate at Precision@50, but the baseline produced the largest practical lift.

### Final method decision

The Week-4 baseline remains the recommended prioritization method for this dataset because it achieved the best Precision@50 on held-out clients.

Random Forest may contain useful broader ranking signal because it achieved the highest ROC-AUC, but it should not replace the baseline based on these results. A future experiment could evaluate a hybrid approach in which the Week-4 eligibility rule defines the candidate set and a learned model reranks only the eligible rows.

The results are predictive and associational. They do not demonstrate that low CTR, content age, search position, or any other feature causes future performance decline.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.